# Hala H36 paired no-numeric-example prompt ablation

32 existing exposed first-party H34 synthetic DEV images; two matched prompt arms, 64 inferences in one Kaggle T4 Batch VM. Same model, frames and evaluation, alternating arm order. **No independent held-out TEST or real GUI qualification.** No user or device actions. Zero-cost, noncommercial research only.


In [ ]:
import hashlib, json, os, platform, subprocess, time
from pathlib import Path
H36_BOOTSTRAP_START = time.monotonic()
source_dir = Path('/kaggle/working/hala-h36-immutable-src')
output_dir = Path('/kaggle/working/hala-h36-paired-output')
os.environ['HALA_H36_RESEARCH_ALLOWED'] = '1'
if os.environ['HALA_H36_RESEARCH_ALLOWED'] != '1':
    raise RuntimeError('explicit H36 noncommercial research opt-in required')
if platform.system() != 'Linux' or not Path('/kaggle/working').is_dir():
    raise RuntimeError('only Kaggle Linux allowed')
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') != 'Batch':
    raise RuntimeError('retained Batch is required')
if source_dir.exists() or output_dir.exists():
    raise FileExistsError('cannot clobber previous H36 source/evidence')
os.environ.update(HF_HOME='/tmp/hala-h34-cache', HF_HUB_DISABLE_TELEMETRY='1',
                  HF_HUB_DISABLE_IMPLICIT_TOKEN='1', DO_NOT_TRACK='1',
                  WANDB_DISABLED='true', TOKENIZERS_PARALLELISM='false')
subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--single-branch',
                '--branch', 'ps/h36-paired-prompt-ablation', 'https://github.com/TheHalfMoon/Hala.git',
                str(source_dir)], check=True, timeout=300)
def git(*args):
    return subprocess.check_output(['git', *args], cwd=source_dir, text=True, timeout=25).strip()
if git('rev-parse', 'HEAD') != 'ae3f684580c5cd3fba63442738b945de88ad1085':
    raise RuntimeError('source commit changed; abort run')
if git('status', '--porcelain', '--untracked-files=all'):
    raise RuntimeError('remote source is dirty')
source_manifest = json.loads('{"h34/DEJAVU_LICENSE.txt": "7a083b136e64d064794c3419751e5c7dd10d2f64c108fe5ba161eae5e5958a93", "h34/__init__.py": "82b0ab3ca5ab8e7403cf4c6c27026ac39b11ac1b077b9d652d2036c24dd444e0", "h34/audit.py": "b29a3b46764012189c9012e2ae8adb17913f3c57eb8a6f0d84864cc8399b7ace", "h34/corpus.py": "45560479bb8cfd686a92b8735bdae41a72fda26aaafeecdf8825a7e6a0a8d667", "h34/font_pin.json": "4d3ce302f0dfa35995cb4e7c972457b1f87a4e07f0fe9665b170ded8f8eecdc1", "h34/frozen/cases.json": "4b7f849a624caac23dd857adb19ed4293d93d0693dbdbd2921d4423b666a6a3a", "h34/geometry.py": "e430e55c5390a8149e6c3a1fd6cba40a603001d8b927afd24adfe560a35ec7d4", "h34/protocol.py": "274bb4d3694771288b2777731867f5ebfc13bae47c939f72d1d63b097e27869c", "h34/render.py": "e2279e09da4e845990be1268a5efdf4e7962cf10cff7a4c382606ac1350fd2ff", "h34/runner.py": "155ca719d01da7a00dc6f8b7886d48105b83e79f17684848ac7c6670bcb55e0c", "h36/__init__.py": "29dd1a2703c823ed59410e58fea7c0aabb62cc59347fa53f82c28ad28336f0fe", "h36/audit.py": "93f7bb8c90b7e0f64ac2d58bd8b47581f62e3beb2de70c1524a2498f042872b0", "h36/protocol.py": "e00e9d135ca37ffdac34d4536ff21ace11f7704195afba922da5e5d0833ef58f", "h36/runner.py": "dbbed24d025fe0f85c302fdbd7ec16bb6c98a87637e147ab96c386d12bae5055", "tools/build_h34_kaggle.py": "6e0375198e503fc233a3421d8c78b7fe347838d9d52223ea9a9fcdda979e75e8", "tools/build_h36_kaggle.py": "c3fd7c0d526c22381dadb818629db0c0d30bd0c8c95a3a1106508e1374daf9fb"}')
for name, expected in source_manifest.items():
    if hashlib.sha256((source_dir / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('signed source file mismatch: ' + name)
print('H36_SHA_BOUND_SOURCE', 'ae3f684580c5cd3fba63442738b945de88ad1085', 'FILES_SHA256', '1a859e5874baeba4cf8c14de1fae8df74bf806ac56688b032a8f54296ef5b48b')


In [ ]:
import importlib.metadata
print('H36_PINNED_DEPS', {name: importlib.metadata.version(name) for name in ('torch','transformers','accelerate','safetensors','Pillow')})


In [ ]:
import os, signal, subprocess, sys, time
cmd = [sys.executable, '-u', '-m', 'h36.runner',
       '--output', '/kaggle/working/hala-h36-paired-output',
       '--source-commit', 'ae3f684580c5cd3fba63442738b945de88ad1085']
proc = subprocess.Popen(cmd, cwd=source_dir, start_new_session=True)
try:
    code = proc.wait(timeout=max(1, min(1500, 1740-(time.monotonic()-H36_BOOTSTRAP_START))))
except subprocess.TimeoutExpired:
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait(timeout=10)
    raise RuntimeError('outer H36 bounded deadline exceeded; retain partial output only')
if code != 0:
    raise RuntimeError('H36 owned process failed; no qualification')
print('H36_REAL_RAW_REQUIRES_SEPARATE_SHA_BOUND_AUDIT')
